# 2 · Safety Stock and Replenishment Policy

<a href="https://colab.research.google.com/github/FH-Prevail/jku_course/blob/main/notebooks/2_forecast_to_stock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

If the button does not open, use this link, or copy it into the browser: <a href="https://colab.research.google.com/github/FH-Prevail/jku_course/blob/main/notebooks/2_forecast_to_stock.ipynb" target="_parent">https://colab.research.google.com/github/FH-Prevail/jku_course/blob/main/notebooks/2_forecast_to_stock.ipynb</a>

Or in Colab: **File, Open notebook, GitHub**, type `FH-Prevail/jku_course` and choose the notebook.

*UE Digital Analytics im Handel · JKU Linz · 03.10.2026 · Sina Mirshahi, Logistikum*

Choose **Copy to Drive** (File, Save a copy in Drive), then **Runtime → Run all**. You do not have to write code: run the cells, read the outputs and use the controls.

Every cell shows its code above its result. The first cell loads the course data and the precomputed forecasts from the course repository, github.com/FH-Prevail/jku_course; it does not train a model, and it takes a few seconds.

**Ask Gemini about the code:** select a cell and ask, for example, “Explain this code line by line in plain English” or “Change this to show the Puzzle 1000 instead”. Read what it changed before you run it, and check its answer against the numbers. If Gemini is unavailable, ask a neighbour or the lecturer. Use only the synthetic course data.

**If a control is missing:** use the static example directly above it. If you edited a cell by accident, Undo, then run it again; otherwise reopen a fresh copy from the repository. Restarting does not undo edits.

## Your assignment (graded)

This notebook is **half of your T2 assignment**; notebook 1 is the other half. Together they count **25 % of the course grade**, plus 5 % for taking part in class. This notebook has **3 tasks and 10 points**.

- **Every task says step by step what to do** and shows a worked example for the Plush Bear in store. Do the same for your own product, with your own numbers.
- **Write your answers** in the cells marked **Your answer**: double-click the cell, replace each ___ with your answer, then press Shift + Enter.
- **Individual work.** You may discuss with others, but the answers you submit are your own.
- **Gemini is allowed.** Say in your answer where you used it, and check what it says against the numbers.
- **Submit by Saturday 10 October 2026, 23:59** in the T2 assignment on MS Teams: in Colab choose **File, Download, Download .ipynb**, name the file `T2_Notebook2_Lastname_Firstname.ipynb` and upload it.
- Run all cells before you download, so your results are saved next to your answers.

In [1]:
# Start here: run this cell first. It loads the course data from GitHub and prepares the pictures.
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display

SOURCE = "https://raw.githubusercontent.com/FH-Prevail/jku_course/main/data/classroom"   # the course data on GitHub
demand = pd.read_csv(f"{SOURCE}/demand.csv", parse_dates=["week_start"])          # weekly sales: one row per product, channel and week
products = pd.read_csv(f"{SOURCE}/products.csv").set_index("product_id")           # price, cost and supplier lead time of each product
comparisons = pd.read_csv(f"{SOURCE}/comparisons.csv", parse_dates=["week_start", "test_start"])  # the backtest forecasts
inventory = pd.read_csv(f"{SOURCE}/inventory.csv", parse_dates=["week_start"])     # forecasts 1 to 7 weeks ahead, for the stock replay
results = json.load(urllib.request.urlopen(f"{SOURCE}/results.json"))             # numbers computed in advance

METHODS = ["Repeat last year", "Simple exponential smoothing", "Prophet", "LightGBM in global mode"]  # the four approaches
PRODUCTS = {"Plush Bear, store": "P010_store", "Plush Bear, online": "P010_online",   # name shown -> series in the data
            "Puzzle 1000, store": "P011_store", "Vacuum Filter, store": "P035_store",
            "Garden Hose, store": "P015_store"}
WINDOWS = {"Spring": "2025-03-24", "Summer": "2025-06-16", "Christmas": "2025-12-01"}   # first week of each 4-week test window
COLORS = {"Actual sales": "#262626", "Repeat last year": "#E66A1F", "Simple exponential smoothing": "#1B9E5A",
          "Prophet": "#D04F95", "LightGBM in global mode": "#0B5CAD"}                 # the same colour per method as on the slides
MARKERS = {"Repeat last year": "s", "Simple exponential smoothing": "^", "Prophet": "X", "LightGBM in global mode": "D"}
BLUE = "#0B5CAD"
plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.2, "figure.dpi": 110})


def series_id(product):
    """Turn a product name such as 'Plush Bear, store' into its series in the data, 'P010_store'."""
    if product in PRODUCTS:
        return PRODUCTS[product]
    if product in PRODUCTS.values():
        return product
    raise ValueError("Choose one of the product names in the dropdown.")


def neat(table, formats=None, na_rep=""):
    """Show a table the same way in every cell: all titles in one header row, the first column on the left,
    the other columns on the right, under their titles."""
    table = table.rename_axis(columns=None)          # no title above the column titles
    if table.index.name is not None:
        table = table.reset_index()                  # the row labels become an ordinary first column
    styled = table.style.hide(axis="index")
    if formats:
        styled = styled.format(formats, na_rep=na_rep)
    return styled.set_table_styles([
        {"selector": "th, td", "props": "text-align: right; padding: 4px 14px;"},
        {"selector": "th.col0, td.col0", "props": "text-align: left;"},
    ])

print("Ready. The course data is loaded.")
print("The forecasts were computed in advance, using only what was known at each forecast date.")


Ready. The course data is loaded.
The forecasts were computed in advance, using only what was known at each forecast date.


**Your name:** ___

**Student number:** ___

## 1. A forecast is not an order

**Lead time** is the wait between placing and receiving an order. We review stock every week; the **protection period** covers lead time plus that review week.

**Safety stock** is a cushion for uncertain demand. **Stock target**, also called the order up to level, is forecast demand over the protection period plus safety stock. **Order now** is the shortfall between that target and stock already on the shelf or on order, assuming no backorders.

This worked example uses round teaching numbers: five forecasts of 180, 190, 210, 200 and 220 units. The forecasts sum to 1,000; the 95 percent planning target adds 148 units of safety stock.

In [2]:
def order_example(target=95):
    """The worked order from the slides, with round teaching numbers."""
    forecasts = np.array([180, 190, 210, 200, 220])     # the five weeks the stock must cover
    weekly_sigma, lead_time = 40.0, 4                    # error spread per week; supplier lead time in weeks
    safety_stock = int(np.ceil(norm.ppf(target / 100) * weekly_sigma * np.sqrt(lead_time + 1)))
    stock_target = int(forecasts.sum() + safety_stock)
    on_hand, on_order = 300, 500
    order = max(0, stock_target - on_hand - on_order)
    return {"weekly_forecasts": forecasts.tolist(), "forecast_total": int(forecasts.sum()), "weekly_sigma": weekly_sigma,
            "protection_weeks": lead_time + 1, "safety_stock": safety_stock, "stock_target": stock_target,
            "on_hand": on_hand, "on_order": on_order, "order": order}


def worked_order(target=95):
    """Show the worked order as a table, for a cycle service target in percent."""
    e = order_example(target)
    print(f"Teaching scenario: five weekly forecasts, one order decision, a {target} percent service target. These are round illustrative numbers.")
    display(neat(pd.DataFrame({"Quantity": ["Forecast over five weeks", "Safety stock", "Stock target", "Already on the shelf", "Already on order", "Order now"],
                               "Units": [e["forecast_total"], e["safety_stock"], e["stock_target"], e["on_hand"], e["on_order"], e["order"]]})))
    print("We assume no backorders and arrivals as scheduled. The stock target is not the quantity to order.")


worked_order()

Teaching scenario: five weekly forecasts, one order decision, a 95 percent service target. These are round illustrative numbers.


Quantity,Units
Forecast over five weeks,1000
Safety stock,148
Stock target,1148
Already on the shelf,300
Already on order,500
Order now,348


We assume no backorders and arrivals as scheduled. The stock target is not the quantity to order.


**What to see.** The stock target is 1,148 units, but the order is 348 because 800 units are already on the shelf or on order. Ordering the whole target would count that stock twice.

### Task 2.1 · The worked order at 99 percent (2 points)

**What to do**
1. In the cell below, change **95** to **99** and run it. It shows the same table as above, for a 99 percent target.
2. Copy three numbers into the answer table: the safety stock, the stock target and the order now.
3. Complete the sentence under the table.

In [3]:
worked_order(95)   # change 95 to 99

Teaching scenario: five weekly forecasts, one order decision, a 95 percent service target. These are round illustrative numbers.


Quantity,Units
Forecast over five weeks,1000
Safety stock,148
Stock target,1148
Already on the shelf,300
Already on order,500
Order now,348


We assume no backorders and arrivals as scheduled. The stock target is not the quantity to order.


**Your answer, Task 2.1**

| | At 95 % (the table above) | At 99 % (your result) |
|---|---|---|
| Safety stock, units | 148 | ___ |
| Stock target, units | 1,148 | ___ |
| Order now, units | 348 | ___ |

- The order is smaller than the stock target because ___ units are already on the shelf or on order; ordering the whole target would ___.

**Where the safety stock comes from** (slide 51). The safety factor z belongs to the bell curve: for a 95 percent target, demand stays below the forecast plus 1.645 standard deviations in 95 of 100 order cycles. Multiply z by the error spread over the five weeks to get the safety stock.

In [4]:
def z_table(targets=(50, 84.13, 90, 95, 99)):
    """Where the safety stock of the worked order comes from (slide 51): z for each target, times the error spread over five weeks."""
    e = order_example()
    spread = e["weekly_sigma"] * np.sqrt(e["protection_weeks"])        # 40 x √5 = 89.4 units
    table = []
    for t in targets:                                                    # 84.13 % is exactly one standard deviation
        z = norm.ppf(t / 100)
        table.append([f"{t:.0f} %", f"{z:.3f}" if t == 95 else f"{z:.2f}", int(np.ceil(z * spread))])
    print(f"Error spread (sd) over the {e['protection_weeks']} weeks: {e['weekly_sigma']:.0f} x √{e['protection_weeks']} = {spread:.1f} units.")
    print("Safety stock = z x that spread, rounded up to whole units.")
    return neat(pd.DataFrame(table, columns=["Cycle service target", "z", "Safety stock, units"]))


z_table()

Error spread (sd) over the 5 weeks: 40 x √5 = 89.4 units.
Safety stock = z x that spread, rounded up to whole units.


Cycle service target,z,"Safety stock, units"
50 %,0.00,0
84 %,1.00,90
90 %,1.28,115
95 %,1.645,148
99 %,2.33,209


**What to see.** Each step up the curve costs more stock for less extra safety: from 95 to 99 percent the cushion grows by 61 units, about 41 percent. The 84 percent row is exactly one standard deviation (84.13 percent).

## 2. Choose a service target

A **cycle service target** is the share of order cycles, here weeks, that we plan to get through without a stock-out. A target of 95 percent is a planning assumption, not a promise about the percentage of units served.

**Fill rate** is the share of demand actually served. The replay below reports fill rate, weeks with a stock-out and average stock. We do not compare fill rate directly with the cycle service target.

LightGBM in global mode is trained once, before 2025. The error spread (sd, the standard deviation of the weekly misses) is measured on 25 weeks from January to June. Orders are rounded up to whole units.

**The replay** is like the backtest, but for stock: we run the ordering rule week by week on the real demand of 30 June to 22 December 2025, as if we had used it then. Each week it orders up to the stock target, the delivery arrives after the lead time, customers buy what is on the shelf, and a sale we miss is lost. Then we count the average stock, the fill rate and the weeks with a stock-out.

In [5]:
def profile(product="Plush Bear, store", method="LightGBM in global mode"):
    """One product's 2025 weeks for the stock replay: actual sales, the forecasts for the weeks the stock must cover,
    and the error spread measured on the first 25 weeks."""
    sid = series_id(product)
    if method not in METHODS:
        raise ValueError("Choose a method shown in the table.")
    weeks = inventory[(inventory["series_id"] == sid) & (inventory["method"] == method)].sort_values("week_start")
    if weeks.empty:
        raise ValueError("This product has no stock exercise.")
    actual = weeks["actual"].to_numpy(float)
    forecasts = weeks[[f"h{h}" for h in range(1, 8)]].to_numpy(float)     # forecasts 1 to 7 weeks ahead, made each week
    lead_time = int(products.loc[sid.split("_")[0], "lead_time_weeks"])
    forecasts = forecasts[:, :lead_time + 1]                                # keep the weeks until the next delivery
    first_half = 25                                                         # January to June: measure the error spread
    sigma = float(np.std(actual[:first_half] - forecasts[:first_half, 0], ddof=1))
    return weeks, actual, forecasts, sigma, lead_time, first_half


def simulate(product="Plush Bear, store", target=95, method="LightGBM in global mode", details=False):
    """Replay the ordering rule week by week over the second half of 2025 and measure what it achieved."""
    if not np.isfinite(target) or not 50 <= target <= 99:
        raise ValueError("Choose a target from 50 to 99 percent.")
    weeks, actual, forecasts, sigma, lead_time, start = profile(product, method)
    safety_stock = float(norm.ppf(target / 100) * sigma * np.sqrt(lead_time + 1))
    on_hand = float(np.ceil(forecasts[start].sum()))    # begin with one protection period of stock on the shelf
    pipeline = []                                        # orders on their way: (week they arrive, units)
    trace = []
    for t in range(start, len(actual)):
        on_hand += sum(units for arrives, units in pipeline if arrives == t)        # this week's deliveries arrive
        pipeline = [(arrives, units) for arrives, units in pipeline if arrives != t]
        stock_target = float(np.ceil(forecasts[t].sum() + safety_stock))           # forecast over the protection period + safety stock
        order = max(0.0, stock_target - on_hand - sum(units for _, units in pipeline))
        pipeline.append((t + lead_time, order))
        served = min(on_hand, actual[t])                 # we can only sell what is on the shelf
        lost = actual[t] - served                        # no backorders: unserved demand is lost
        on_hand -= served
        trace.append({"week": str(weeks["week_start"].iloc[t].date()), "actual": float(actual[t]),
                      "forecast_sum": float(forecasts[t].sum()), "level": stock_target, "order": order, "lost": lost, "stock": on_hand})
    replay = pd.DataFrame(trace)
    total = replay["actual"].sum()
    result = {"target": float(target), "safety_stock": float(np.ceil(safety_stock)),
              "average_stock": float(replay["stock"].mean()),
              "fill_rate": float(100 * (1 - replay["lost"].sum() / total)) if total else 100.0,
              "stockout_weeks": int((replay["lost"] > 0).sum()), "lost_units": float(replay["lost"].sum()),
              "weeks_stock": float(replay["stock"].mean() / replay["actual"].mean()) if total else 0.0,
              "weekly_sigma": sigma, "protection_weeks": lead_time + 1, "test_weeks": len(replay)}
    return (result, trace) if details else result


def stock_decision(product="Plush Bear, store", target=95):
    """What one service target achieved in the replay."""
    r = simulate(product, target)
    lead_time = profile(product)[4]
    print(f"{product}: cycle service target {target} %; supplier lead time {lead_time} weeks.")
    display(neat(pd.DataFrame({"Measure": ["Average stock on the shelf", "Fill rate: share of demand served", "Weeks with a stock-out", "Demand not served"],
                               "Observed result": [f"{r['average_stock']:.0f} units", f"{r['fill_rate']:.1f} %",
                                                   f"{r['stockout_weeks']} of {r['test_weeks']} weeks", f"{r['lost_units']:.0f} units"]})))
    print(f"Safety stock allowance: about {r['safety_stock']:.0f} units. This is part of the stock target, not the order quantity.")
    print("A cycle service target is a planning probability; fill rate measures units served. They are different measures.")


stock_decision('Plush Bear, store', 95)

Plush Bear, store: cycle service target 95 %; supplier lead time 4 weeks.


Measure,Observed result
Average stock on the shelf,289 units
Fill rate: share of demand served,100.0 %
Weeks with a stock-out,0 of 26 weeks
Demand not served,0 units


Safety stock allowance: about 147 units. This is part of the stock target, not the order quantity.
A cycle service target is a planning probability; fill rate measures units served. They are different measures.


**Try it.** Move Target % from 50 to 80, then to 95 and 99. Does every increase serve more demand? Then choose Puzzle 1000 and Vacuum Filter; the same target need not deliver the same observed outcome.

In [ ]:
def stock_widget():
    """A product dropdown and a target slider that rerun the replay."""
    from ipywidgets import interactive, Dropdown, IntSlider
    controls = interactive(stock_decision,
                           product=Dropdown(options=["Plush Bear, store", "Puzzle 1000, store", "Vacuum Filter, store"], description="Product"),
                           target=IntSlider(value=95, min=50, max=99, step=1, description="Target %", continuous_update=False))
    display(controls)
    return controls


stock_controls = stock_widget()

**If the slider is unavailable**, compare these preset settings instead.

In [7]:
def stock_table(product="Plush Bear, store", targets=(50, 80, 90, 95, 99)):
    """The replay at five service targets, side by side."""
    table = []
    for t in targets:
        r = simulate(product, t)
        table.append([f"{t} %", f"{r['average_stock']:.0f}", f"{r['fill_rate']:.1f} %", r["stockout_weeks"], f"{r['lost_units']:.0f}"])
    return neat(pd.DataFrame(table, columns=["Cycle service target", "Average stock, units", "Fill rate", "Weeks with a stock-out",
                                             "Units not served"]))


stock_table('Plush Bear, store')

Cycle service target,"Average stock, units",Fill rate,Weeks with a stock-out,Units not served
50 %,181,98.0 %,7,85
80 %,228,100.0 %,0,0
90 %,261,100.0 %,0,0
95 %,289,100.0 %,0,0
99 %,340,100.0 %,0,0


**What to see.** Compare each increase in stock with the change in fill rate. If fill rate is unchanged, the extra stock served no additional demand in these test weeks; this does not prove that the extra cushion is useless in every future period.

### Task 2.2 · What does each step buy? (4 points)

**What to do**
1. Choose **your product**: `Puzzle 1000, store` or `Vacuum Filter, store`. You use it in Task 2.3 too.
2. In the cell below, put its name (keep the quotation marks) and run it. The table shows the replay at five targets.
3. Copy the rows for **80, 90, 95 and 99 %** into the answer table; skip 50 %.
4. Compare the 95 % row with the 99 % row and complete the two sentences.

**Worked example, Plush Bear in store** (the table above): from 95 to 99 % the average stock rises from 289 to 340 units, and the fill rate goes from 100.0 % to 100.0 %. The extra 51 units served no more customers in these weeks, so the step to 99 % is not worth it for the Bear.

In [8]:
stock_table("Puzzle 1000, store")   # or "Vacuum Filter, store"

Cycle service target,"Average stock, units",Fill rate,Weeks with a stock-out,Units not served
50 %,28,59.5 %,11,357
80 %,41,75.5 %,9,216
90 %,53,82.2 %,7,157
95 %,65,87.2 %,6,113
99 %,91,92.4 %,5,67


**Your answer, Task 2.2**

My product: ___

| Target | Average stock, units | Fill rate | Weeks with a stock-out |
|---|---|---|---|
| 80 % | ___ | ___ | ___ |
| 90 % | ___ | ___ | ___ |
| 95 % | ___ | ___ | ___ |
| 99 % | ___ | ___ | ___ |

- From 95 to 99 %, the average stock rises from ___ to ___ units (what it costs), and the fill rate goes from ___ % to ___ % (what it buys).
- Is the step to 99 % worth it for this product? ___ (yes or no), because ___

## 3. Compare the products

All three products below use a 95 percent cycle service target. “Weeks of demand on shelf” divides average stock by average demand during the same replay period.

In [9]:
def stock_compare():
    """Three products, the same 95 percent target."""
    table = []
    for product in ["Plush Bear, store", "Puzzle 1000, store", "Vacuum Filter, store"]:
        r = simulate(product, 95)
        table.append([product, f"{r['fill_rate']:.1f} %", r["stockout_weeks"], f"{r['weeks_stock']:.1f}"])
    return neat(pd.DataFrame(table, columns=["Product", "Fill rate at 95 % target", "Weeks with a stock-out",
                                             "Weeks of demand on shelf"]))


stock_compare()

Product,Fill rate at 95 % target,Weeks with a stock-out,Weeks of demand on shelf
"Plush Bear, store",100.0 %,0,1.7
"Puzzle 1000, store",87.2 %,6,1.9
"Vacuum Filter, store",93.3 %,1,7.5


**What to see.** A high target can coexist with shortages when the error assumptions are poor. A slow mover can hold many weeks of demand in only a few units; inspect unit cost before calling that stock excessive.

The safety stock calculation assumes independent weekly errors, a stable error distribution and an approximate bell curve. Those assumptions are especially weak for sparse demand. Treat the replay as a check on the rule, not a service guarantee.

## 4. Your management recommendation

### Task 2.3 · Your stock recommendation (4 points)

Imagine the store manager asks you: *"Which service target should we use for this product?"* Answer in five short lines, for your product from Task 2.2.

**What to do**
1. Write your product.
2. **Choose one target:** 80, 90, 95 or 99 %.
3. **Give your evidence:** at that target, the average stock and the fill rate from your Task 2.2 table.
4. **Say what you give up**, and explain it in one sentence with your numbers:
   - **a.** More stock tied up on the shelf.
   - **b.** Some customers not served.
   - **c.** Less safety in unusual weeks.
5. **Name one check** before using it in the real store, and say why:
   - **a.** What holding one unit for a year costs.
   - **b.** What a lost sale costs.
   - **c.** Whether the forecast runs too high or too low (its bias).

**Worked example, Plush Bear in store:** (1) Plush Bear, store. (2) 90 %. (3) The shelf held on average 261 units and served 100.0 % of demand. (4) Give up c: less safety in unusual weeks; but in these 26 weeks 95 and 99 % served no more customers and held 28 and 79 more units. (5) Check b: what a lost sale at Christmas costs, because one empty-shelf week could cost more than the stock we save.

**Your answer, Task 2.3**

1. Product: ___
2. Target: ___ %
3. My evidence: at this target the shelf held on average ___ units and served ___ % of demand.
4. What I give up (a, b or c): ___, because ___
5. Check (a, b or c): ___, because ___

You have completed the graded tasks; the next sections are optional.

## 5. Optional: what is a better forecast worth?

Keep the same 95 percent target and change the forecasting approach. Each method uses its own error estimate. Read stock and service together; lower stock with substantially lower service is a different policy, not automatically a better one.

In [10]:
def forecast_value(product="Plush Bear, store"):
    """The same 95 percent target with each forecast: stock, service and the yearly cost of holding the stock."""
    item = products.loc[series_id(product).split("_")[0]]
    table = []
    for method in METHODS:
        r = simulate(product, 95, method)
        cost = r["average_stock"] * item["unit_cost"] * item["holding_cost_rate"]
        table.append([method, f"{r['average_stock']:.0f}", f"{r['fill_rate']:.1f} %", f"EUR {cost:.0f}"])
    print(f"{product}. Same 95 % target, each method uses its own measured errors.")
    print(f"Unit cost EUR {item['unit_cost']:.2f}; annual holding rate {item['holding_cost_rate']:.0%}.")
    print("Annualised stock holding cost only: a scenario based on the replay average, excluding shortages and implementation.")
    return neat(pd.DataFrame(table, columns=["Method", "Average stock, units", "Fill rate", "Annualised holding cost"]))


forecast_value()

Plush Bear, store. Same 95 % target, each method uses its own measured errors.
Unit cost EUR 13.83; annual holding rate 20%.
Annualised stock holding cost only: a scenario based on the replay average, excluding shortages and implementation.


Method,"Average stock, units",Fill rate,Annualised holding cost
Repeat last year,351,100.0 %,EUR 971
Simple exponential smoothing,208,88.0 %,EUR 574
Prophet,225,100.0 %,EUR 623
LightGBM in global mode,289,100.0 %,EUR 798


**What to see.** The last column prices the average stock using this product’s unit cost and a 20 percent annual holding rate. It is an annualised scenario based on a half-year replay, not realised annual savings or a complete investment case; shortages and implementation costs are excluded.

## 6. Optional: one Christmas order

The **newsvendor** problem is one order for one selling period. A unit short loses margin; a leftover unit costs its purchase price minus what we recover afterward.

The **critical ratio** is shortage cost divided by shortage cost plus leftover cost. It chooses a **quantile**: a demand level with a specified probability of not being exceeded. Here demand is approximated with a bell curve, so the 50th percentile is its mean.

This is a separate decision assuming the order can arrive for the selling week; it does not use the replenishment lead time. Use recovery below 100 percent: with no leftover cost, this unbounded model has no finite optimal order.

In [11]:
def christmas_order(salvage=50):
    """One order for one selling week (the newsvendor): the order is a quantile of demand set by the critical ratio."""
    if not np.isfinite(salvage) or not 0 <= salvage <= 95:
        raise ValueError("Choose recovery from 0 to 95 percent.")
    weeks, actual, forecasts, sigma, lead_time, first_half = profile("Puzzle 1000, store")
    i = np.flatnonzero(weeks["week_start"] == pd.Timestamp("2025-12-15"))[0]
    item = products.loc["P011"]
    cost_short = float(item["unit_price"] - item["unit_cost"])             # lost margin on a unit we did not have
    cost_leftover = float(item["unit_cost"] * (1 - salvage / 100))          # purchase cost minus what we recover
    ratio = cost_short / (cost_short + cost_leftover)                       # the critical ratio
    order = max(0, int(np.ceil(forecasts[i, 0] + norm.ppf(ratio) * sigma)))
    return {"forecast": float(forecasts[i, 0]), "sigma": sigma, "under": cost_short, "over": cost_leftover,
            "ratio": ratio, "order": order, "actual": float(actual[i])}


def christmas_display(salvage=50):
    """Print the Christmas order in words."""
    r = christmas_order(salvage)
    print(f"Puzzle 1000, store, 15 December 2025: forecast {r['forecast']:.0f} units; actual {r['actual']:.0f}.")
    print(f"Recover {salvage} % of purchase cost on leftovers: a unit short costs EUR {r['under']:.2f}, a leftover costs EUR {r['over']:.2f}.")
    print(f"Critical ratio {r['ratio']:.2f}: choose that quantile of demand. Suggested order {r['order']} units.")
    print("A quantile is a demand level with a chosen probability of not being exceeded. Here we approximate demand with a bell curve.")
    print("This separate exercise assumes one order can arrive for the selling week; it does not simulate the supplier lead time.")


christmas_display(50)

Puzzle 1000, store, 15 December 2025: forecast 88 units; actual 41.
Recover 50 % of purchase cost on leftovers: a unit short costs EUR 23.23, a leftover costs EUR 19.43.
Critical ratio 0.54: choose that quantile of demand. Suggested order 92 units.
A quantile is a demand level with a chosen probability of not being exceeded. Here we approximate demand with a bell curve.
This separate exercise assumes one order can arrive for the selling week; it does not simulate the supplier lead time.


In [ ]:
def christmas_widget():
    """A slider for the share of the purchase cost recovered on leftovers."""
    from ipywidgets import interactive, IntSlider
    controls = interactive(christmas_display,
                           salvage=IntSlider(value=50, min=0, max=95, step=5, description="Recovery %", continuous_update=False))
    display(controls)
    return controls


christmas_controls = christmas_widget()

**What to see.** Higher recovery makes leftovers cheaper and increases the order, while the forecast stays fixed. This shows why a forecast alone cannot choose an order quantity.

**Take home.** Keep the forecast, its error and the business cost of a mistake together. Your recommendation should state a choice, evidence and a limitation.

## Before you submit

1. **Runtime, Run all**, so every result is saved next to your answers.
2. Check that every **Your answer** cell is filled in: no ___ left.
3. **File, Download, Download .ipynb**. Name the file `T2_Notebook2_Lastname_Firstname.ipynb`.
4. Upload it to the T2 assignment on MS Teams by **Saturday 10 October 2026, 23:59**.